In [ ]:
import sys
import pandas as pd
import numpy as np
import torch
import ast
import pickle
import torch

import networkx as nx
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from src.ml_models import *

In [ ]:
seed = 13
random.seed(seed)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

num_workers = 4 if sys.platform == "linux" else 0
print(f"Num workers: {num_workers}")

In [ ]:
with open("gen/network.pickle", "rb") as f:
    G = pickle.load(f)

In [ ]:
df = pd.read_csv("gen/ml_data.csv")
df["defect_path"] = df["defect_path"].apply(ast.literal_eval)
df.head()

## Apply logarithmic scaling to target values

In [ ]:
df["task_timespent"] = np.log(df["task_timespent"])
df.head()

### Split dataset into training and validation sets

In [ ]:
X = []
y = []

for _, row in df.iterrows():
    path_template = unpack_path(row["defect_path"])
    cost = row["task_timespent"]
    X.append(path_template)
    y.append(cost)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=seed)

### Model Training

In [ ]:
MODEL_TYPE = "GRU"
#MODEL_TYPE = "RNN"

if MODEL_TYPE == "GRU":
    hidden_size = 2
    num_layers = 2
    base_model = GRUModel(
        input_size=DIMENSIONS, 
        hidden_size=hidden_size, 
        output_size=1, 
        num_layers=num_layers, 
        dropout=0.2
    )
elif MODEL_TYPE == "RNN":
    hidden_size = 3
    num_layers = 2
    base_model = RNNModel(
        input_size=DIMENSIONS, 
        hidden_size=hidden_size, 
        output_size=1, 
        num_layers=num_layers, 
        dropout=0.2
    )
else:
    print("Model type must be defined.")
    raise

total_parameters = sum(p.numel() for p in base_model.parameters())
print(f"Number of Parameters: {total_parameters}")

In [ ]:
TRAINING_ITERATIONS = 25

training_results = []

for i in range(1, TRAINING_ITERATIONS + 1):
    print(f"Training Iteration: {i}/{TRAINING_ITERATIONS}")
    results_dict = train_em_model(
        model=base_model,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        G=G,
        embeddings=embeddings,  # embeddings imported from ml_models.py
        max_epochs=5000,
        device=device,
        patience=20,
        init_epochs=50,
        m_step_epochs=20,
        m_patience=10,
    )

    training_results.append(results_dict)
    print("\n---\n")

In [ ]:
# Select most performant model
losses = [tr["best_loss"] for tr in training_results]
best_idx = np.argmin(losses)
best_result = training_results[best_idx]

model = best_result["model"]
best_X_train = best_result["X_train"]
best_X_test = best_result["X_test"]

## Results

In [ ]:
TITLE_PARAMS = {"fontsize": "14", "fontweight": "semibold"}
LABEL_PARAMS = {"fontsize": "12"}

cm = plt.cm.viridis(np.linspace(0, 1, 5))

In [ ]:
model.eval()

train_dataset = EMDataset(best_X_train, y_train, embeddings)
test_dataset = EMDataset(best_X_test, y_test, embeddings)

train_loader = DataLoader(train_dataset, batch_size=32, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, collate_fn=collate_fn)

train_preds, train_targets = [], []
test_preds, test_targets = [], []

with torch.no_grad():
    # Evaluate training data
    for batch_seqs, batch_lengths, batch_targets in train_loader:
        batch_seqs = batch_seqs.to(device)
        preds = model(batch_seqs, batch_lengths)

        train_preds.extend(preds.cpu().tolist())
        train_targets.extend(batch_targets.tolist())

    # Evaluate testing data
    for batch_seqs, batch_lengths, batch_targets in test_loader:
        batch_seqs = batch_seqs.to(device)
        preds = model(batch_seqs, batch_lengths)

        test_preds.extend(preds.cpu().tolist())
        test_targets.extend(batch_targets.tolist())

train_preds = np.exp(train_preds)
train_targets = np.exp(train_targets)

test_preds = np.exp(test_preds)
test_targets = np.exp(test_targets)

plt.figure(figsize=(6,6))

all_targets = np.concatenate([train_targets, test_targets])
all_preds = np.concatenate([train_preds, test_preds])
min_val = 0
max_val = max(all_targets.max(), all_preds.max())

# Plot prediction line
plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    color="gray",
    linestyle="--",
    label="Perfect Prediction",
)

# Plot training set
plt.scatter(
    train_targets,
    train_preds,
    alpha=0.7,
    color=cm[2],
    marker="o",
    label="Training Predictions",
)

# Plot validation set
plt.scatter(
    test_targets,
    test_preds,
    alpha=0.7,
    color=cm[0],
    marker="^",
    label="Validation Predictions",
)

plt.xlabel("Actual Time Spent (hours)", **LABEL_PARAMS)
plt.ylabel("Predicted Time Spent (hours)", **LABEL_PARAMS)
plt.title(f"Time-To-Fix Prediction Results: {MODEL_TYPE}", **TITLE_PARAMS)
plt.grid(True, linestyle=":", alpha=0.7)
plt.legend()
plt.tight_layout()
plt.savefig(f"fig_{MODEL_TYPE.lower()}_pred_results.png")
plt.show()

### Plot Goodness-of-Fit

In [ ]:
from src.functions import (
    get_total_variation_distance,
    get_chi_squared_distance,
    get_kullback_leibler_divergence,
    expand_template,
)


def get_path_counts(paths):
    path_counts = {}
    for nodes in paths:
        if list(nodes)[0] != "SPAWN":
            path = ["SPAWN"] + list(nodes) + ["DETECTION"]
        else:
            path = list(nodes)
        path_str = str(path)
        if path_str in path_counts.keys():
            path_counts[path_str] += 1
        else:
            path_counts[path_str] = 1
    return path_counts


def get_goodness_of_fit_metrics(paths, all_graph_paths, true_probs):
    num_samples = len(paths)
    path_counts = get_path_counts(paths)
    sample_probs = []
    for path in all_graph_paths:
        # Get observed probability from samples
        path_str = str(path)
        if path_str in path_counts.keys():
            sample_count = path_counts[path_str]
        else:
            sample_count = 0
        sample_prob = sample_count / num_samples
        sample_probs.append(sample_prob)

    d_tv = get_total_variation_distance(sample_probs, true_probs)
    d_x2 = get_chi_squared_distance(sample_probs, true_probs)
    d_kl = get_kullback_leibler_divergence(sample_probs, true_probs)

    return_dict = {
        "tv": d_tv,
        "x2": d_x2,
        "kl": d_kl,
    }
    return return_dict


all_graph_paths = [list(p) for p in nx.all_simple_paths(G, "SPAWN", "DETECTION")]

true_probs = []
for path in all_graph_paths:
    # Get true probability from graph
    true_prob = get_path_probability(path, G)
    true_probs.append(true_prob)


metrics = get_goodness_of_fit_metrics(
    best_X_train + best_X_test, all_graph_paths, true_probs
)

print(f"Total Variation Distance:        {metrics['tv']:.3f}")
print(f"Chi-Squared Distance:            {metrics['x2']:.3f}")
print(f"Kullback-Leibler Divergence:     {metrics['kl']:.3f}")

In [ ]:
REGENERATE_GOF_BOUNDS = False

if not REGENERATE_GOF_BOUNDS:
    upper_bounds = {
        "tv": np.float64(0.6669516346549421),
        "x2": np.float64(7.093895516487798),
        "kl": np.float64(1.3183197118019765),
    }

    lower_bounds = {
        "tv": np.float64(0.32304931227931166),
        "x2": np.float64(0.8630166115943259),
        "kl": np.float64(0.45395436163929503),
    }
else:
    upper_bounds = {
        "tv": -np.inf,
        "x2": -np.inf,
        "kl": -np.inf,
    }

    lower_bounds = {
        "tv": np.inf,
        "x2": np.inf,
        "kl": np.inf,
    }

    expanded_dataset = [expand_template(t) for t in X]
    num_samples = len(X)

    for _ in range(10**6):
        hardened_X = list(random.choice(paths) for paths in expanded_dataset)

        path_counts = {}
        for path in hardened_X:
            path_str = str(list(path))
            if path_str in path_counts.keys():
                path_counts[path_str] += 1
            else:
                path_counts[path_str] = 1

        sample_probs = []
        for path in all_graph_paths:
            # Get observed probability from randomly hardened samples
            path_str = str(path)
            if path_str in path_counts.keys():
                sample_count = path_counts[path_str]
            else:
                sample_count = 0
            sample_prob = sample_count / num_samples
            sample_probs.append(sample_prob)

        temp_d_tv = get_total_variation_distance(sample_probs, true_probs)
        temp_d_x2 = get_chi_squared_distance(sample_probs, true_probs)
        temp_d_kl = get_kullback_leibler_divergence(sample_probs, true_probs)

        current_hardening = {
            "tv": temp_d_tv,
            "x2": temp_d_x2,
            "kl": temp_d_kl,
        }

        for dist_type, value in current_hardening.items():
            if value < lower_bounds[dist_type]:
                lower_bounds[dist_type] = value
            elif value > upper_bounds[dist_type]:
                upper_bounds[dist_type] = value

print("Upper Goodness-of-Fit Bounds")
print(upper_bounds)
print("\nLower Goodness-of-Fit Bounds")
print(lower_bounds)

In [ ]:
all_model_metrics = []

for result in training_results:
    X_combined = result["X_train"] + result["X_test"]
    metrics = get_goodness_of_fit_metrics(X_combined, all_graph_paths, true_probs)

    for metric, value in metrics.items():
        if value > upper_bounds[metric]:
            upper_bounds[metric] = value
        elif value < lower_bounds[metric]:
            lower_bounds[metric] = value

    all_model_metrics.append(metrics)

metrics_keys = ["tv", "x2", "kl"]
metrics_labels = [
    r"$d_{TV}$",
    r"$d_{\chi^2}$",
    r"$d_{KL}$",
]
fig, axs = plt.subplots(3, 1, figsize=(6,6), sharex=False)

fig.suptitle(f"Goodness-of-Fit Metrics: {MODEL_TYPE}", **TITLE_PARAMS)

for i, metric in enumerate(metrics_keys):
    ax = axs[i]

    # Get boundary constraints
    low = lower_bounds[metric]
    high = upper_bounds[metric]

    # Set padding around the boundaries so markers near edges aren't clipped
    padding = (high - low) * 0.01
    ax.set_xlim(low - padding, high + padding)
    ax.set_ylim(-0.5, 0.5)

    # Draw the background horizontal bar representing the total feasible range
    ax.barh(0, width=high - low, left=low, color="#e5e5e5", height=0.15, zorder=1)

    # Plot vertical lines for every model in the dataset
    for idx, model_metric in enumerate(all_model_metrics):
        val = model_metric[metric]

        if idx == best_idx:
            ax.vlines(
                val,
                ymin=-0.4,
                ymax=0.4,
                colors=cm[0],
                linewidth=3.5,
                zorder=3,
                label="Best Model" if i == 0 else "",
            )
        else:
            ax.vlines(
                val,
                ymin=-0.25,
                ymax=0.25,
                colors="gray",
                alpha=0.25,
                linewidth=1.5,
                zorder=2,
                label="Other Models"
                if (i == 0 and idx == (1 if best_idx == 0 else 0))
                else "",
            )

    fig.canvas.draw()
    default_ticks = ax.get_xticks()

    # Filter out default ticks that are too close to the bounds
    tick_threshold = (high - low) * 0.05
    clean_ticks = [
        t
        for t in default_ticks
        if (t - low) > tick_threshold and (high - t) > tick_threshold
    ]

    # Combine, sort, and assign the final tick list
    final_ticks = sorted(list(set([low] + clean_ticks + [high])))
    ax.set_xticks(final_ticks)

    # Format tick labels to visually highlight the bounds (e.g., bolding them)
    labels = []
    for t in final_ticks:
        if np.isclose(t, low) or np.isclose(t, high):
            labels.append(f"b{t:.2f}")
        else:
            labels.append(f"{t:.2g}")

    # Bold the bounds labels using standard matplotlib text formatting
    ax.set_xticklabels(
        [f"$\\bf{float(l[1:]):.2f}$" if l.startswith("b") else l for l in labels]
    )

    # Highlight the bound tick lines visually by adding small outer markers
    ax.vlines(
        [low, high], ymin=-0.12, ymax=0.12, colors="#888888", linewidth=1.5, zorder=2
    )

    # --- Styling & Clean up ---
    ax.set_ylabel(
        metrics_labels[i],
        rotation=0,
        labelpad=0,
        va="center",
        ha="right",
        **LABEL_PARAMS,
    )
    ax.set_yticks([])

    # Remove unnecessary border spines to keep it scannable
    for spine in ["top", "left", "right"]:
        ax.spines[spine].set_visible(False)

    # Soften the bottom axis spine
    ax.spines["bottom"].set_color("#cccccc")
    ax.tick_params(axis="x", colors="#555555", labelsize=9)

axs[0].legend(
    loc="upper right", frameon=False, bbox_to_anchor=(1, 1.4), ncol=2, fontsize=10
)

plt.tight_layout()
plt.savefig(f"fig_{MODEL_TYPE.lower()}_goodness_of_fit.png")
plt.show()

In [ ]:
expanded_dataset = [expand_template(t) for t in X]
total_options = 1
for sample in expanded_dataset:
    total_options *= len(sample)

print(f"Total number of unique hardenings: {total_options:.2e}")

### Plot Validation Losses

In [ ]:
all_val_losses = [tr["val_losses"] for tr in training_results]
min_val_loss = np.inf

plt.figure(figsize=(6,6))
label_added = False
for i, losses in enumerate(all_val_losses):
    if i == best_idx:
        continue
    label_text = "Local Minima" if not label_added else None
    plt.plot(losses, color="gray", alpha=0.25, label=label_text)
    label_added = True

plt.plot(all_val_losses[best_idx], color=cm[0], linewidth=2, label="Best Model")

plt.ylim(0, 0.4)

plt.title(f"Validation Losses: {MODEL_TYPE}", **TITLE_PARAMS)
plt.ylabel("MSE Loss", **LABEL_PARAMS)
plt.xlabel("EM-Iteration", **LABEL_PARAMS)
plt.grid(True, linestyle=":", alpha=0.7)
plt.legend()
plt.tight_layout()
plt.savefig(f"fig_{MODEL_TYPE.lower()}_losses_val.png")
plt.show()

### Plot E-Step Losses

In [ ]:
all_e_losses = [tr["e_losses"] for tr in training_results]

plt.figure(figsize=(6,6))
label_added = False
for i, losses in enumerate(all_e_losses):
    if i == best_idx:
        continue

    label_text = "Local Minima" if not label_added else None
    plt.plot(losses, color="gray", alpha=0.25, label=label_text)
    label_added = True

plt.plot(all_e_losses[best_idx], color=cm[1], linewidth=2, label="Best Model")

plt.ylim(0, 0.4)
plt.title(f"E-Step Losses: {MODEL_TYPE}", **TITLE_PARAMS)
plt.ylabel("MSE Loss", **LABEL_PARAMS)
plt.xlabel("EM-Iteration", **LABEL_PARAMS)
plt.grid(True, linestyle=":", alpha=0.7)
plt.legend()
plt.tight_layout()
plt.savefig(f"fig_{MODEL_TYPE.lower()}_losses_e_step.png")
plt.show()

### Plot M-Step Losses

In [ ]:
all_m_losses = [tr["m_losses"] for tr in training_results]

plt.figure(figsize=(6,6))
label_added = False
for i, losses in enumerate(all_m_losses):
    if i == best_idx:
        continue

    label_text = "Local Minima" if not label_added else None
    plt.plot(losses, color="gray", alpha=0.25, label=label_text)
    label_added = True

plt.plot(all_m_losses[best_idx], color=cm[2], linewidth=2, label="Best Model")

plt.ylim(0, 0.4)
plt.title(f"M-Step Losses: {MODEL_TYPE}", **TITLE_PARAMS)
plt.ylabel("MSE Loss", **LABEL_PARAMS)
plt.xlabel("EM-Iteration", **LABEL_PARAMS)
plt.grid(True, linestyle=":", alpha=0.7)
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(f"fig_{MODEL_TYPE.lower()}_losses_m_step.png")
plt.show()

### Combine best results

In [ ]:
plt.figure(figsize=(6,6))

plt.plot(best_result["val_losses"], color=cm[0], label="Validation Losses")
plt.plot(best_result["e_losses"], color=cm[1], label="E-Step Losses")
plt.plot(best_result["m_losses"], color=cm[2], label="M-Step Losses")

plt.ylim(0, 0.2)

plt.title(f"Best Model Training Losses: {MODEL_TYPE}", **TITLE_PARAMS)
plt.ylabel("MSE Loss", **LABEL_PARAMS)
plt.xlabel("EM-Iteration", **LABEL_PARAMS)
plt.grid(True, linestyle=":", alpha=0.7)
plt.legend()
plt.tight_layout()
plt.savefig(f"fig_{MODEL_TYPE.lower()}_losses_best_model.png")
plt.show()

### Save model

In [ ]:
SAVE = True

if SAVE:
    torch.save(model, f"gen/model_{MODEL_TYPE.lower()}.pt")

In [ ]:
log_str = f"""--- {MODEL_TYPE} structure ---
Embedding dimensions: {DIMENSIONS}
Hidden layer size: {hidden_size}
Number of hidden layers: {num_layers}
Total number of parameters: {total_parameters}
"""

if SAVE:
    with open(f"gen/{MODEL_TYPE.lower()}_structure.txt", "w") as f:
        f.write(log_str)